In [1]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

llm = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.5,
        "max_new_tokens": 200
    }
)

chat_model = ChatHuggingFace(llm=llm)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [20]:
class LLMState(TypedDict):
    question : str
    answer : str

In [ ]:
def llmqa(state : LLMState) -> LLMState:
    question = state['question']
    prompt = "Answer the following question {question}?"
    result = chat_model.invoke(prompt).content
    state['answer'] = result
    return state

In [25]:
graph = StateGraph(llmqa)

graph.add_node("llm_qa", llmqa)

graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)
workflow = graph.compile()

c:\Users\Rahul\OneDrive\Desktop\langgraph\myenv\Lib\site-packages\langgraph\graph\state.py:101: UserWarning: Invalid state_schema: <function llmqa at 0x0000026F44091BC0>. Expected a type or Annotated[type, reducer]. Please provide a valid schema to ensure correct updates.
 See: https://langchain-ai.github.io/langgraph/reference/graphs/#stategraph
  warnings.warn(


In [26]:
initial_state = {'question' : "What is capital of INDIA?"}
final_state = workflow.invoke(initial_state)
print(final_state)

KeyError: 'question'